# Federal Reserve Gold-Relevance Classification

อ่าน Clean Master ทั้งชุด แล้วเพิ่มผลจำแนกลง JSONL รูปแบบใกล้เคียง `thaipbs_classified.jsonl` โดยคง fields ต้นฉบับทั้งหมด

- Input: `fed_press_releases_cleaned/fed_press_releases_clean.jsonl`
- Output: `fed_press_releases_cleaned/fed_press_releases_classified.jsonl`
- ใช้ worker 7 ตัว แยก worker ประจำ API key แต่ละตัว
- สร้าง output JSONL และ progress checkpoint ทันทีเมื่อรัน config; append และ flush/fsync ทุก record
- Resume: อ่าน record ที่สำเร็จใน output เดิม แล้วข้าม record นั้น
- Errors: บันทึกแยกใน `fed_press_releases_classification_errors.jsonl`; progress อยู่ใน `fed_press_releases_classification_progress.json`

**คำเตือน:** การรันจะส่งเนื้อหาข่าวทั้งชุดไป ThaiLLM ผ่าน API และอาจมีค่าใช้จ่าย/ข้อจำกัด rate limit ตรวจสอบ `.env` และนโยบายข้อมูลก่อนรัน

ยังไม่ได้รัน flow จำแนกและ checkpoint เวอร์ชันนี้

In [6]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from threading import Event, Lock
from queue import Empty, Queue
from datetime import datetime, timezone
import hashlib
import json
import os
import re
import time

from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

BASE_DIR = Path(r"D:\Project_Github\gold assisstant\dataset\news\fed")
INPUT_PATH = BASE_DIR / "fed_press_releases_cleaned" / "fed_press_releases_clean.jsonl"
OUTPUT_PATH = BASE_DIR / "fed_press_releases_cleaned" / "fed_press_releases_classified.jsonl"
ERROR_PATH = BASE_DIR / "fed_press_releases_cleaned" / "fed_press_releases_classification_errors.jsonl"
PROGRESS_PATH = BASE_DIR / "fed_press_releases_cleaned" / "fed_press_releases_classification_progress.json"

API_BASE_URL = "http://thaillm.or.th/api/v1"
MODEL = "openthaigpt-thaillm-8b-instruct-v7.2"
PROMPT_VERSION = "fed-gold-relevance-v1.0"
MAX_RETRIES = 4
THAILLM_DELAY = 0.25

load_dotenv(BASE_DIR.parents[2] / ".env")
API_KEY_NAMES = [f"THAILLM_API_KEY{i}" for i in range(1, 8)]
MISSING_API_KEYS = [name for name in API_KEY_NAMES if not os.getenv(name)]
if MISSING_API_KEYS:
    raise EnvironmentError(f"Missing required API keys in .env: {', '.join(MISSING_API_KEYS)}")
API_KEYS = [os.environ[name] for name in API_KEY_NAMES]
if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"Input JSONL not found: {INPUT_PATH}")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.touch(exist_ok=True)
clients = [OpenAI(base_url=API_BASE_URL, api_key=key, timeout=60.0, max_retries=0) for key in API_KEYS]
write_lock = Lock()
stop_event = Event()
completion_queue = Queue()
print(f"API keys loaded: {len(API_KEYS)} | Model: {MODEL}")
print(f"Input: {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")

API keys loaded: 7 | Model: openthaigpt-thaillm-8b-instruct-v7.2
Input: D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned\fed_press_releases_clean.jsonl
Output: D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned\fed_press_releases_classified.jsonl


In [7]:
SYSTEM_PROMPT = r'''# SYSTEM PROMPT — FED GOLD-PRICE RELEVANCE AUDITOR
Version: 1.0

## 1. ROLE AND OBJECTIVE
You are a strict financial-news relevance auditor specializing in the relationship between Federal Reserve announcements and gold prices.

Your task is to evaluate ONE individual Federal Reserve news release at a time and determine whether its actual content is relevant to understanding factors that may influence gold prices.

You are performing news relevance classification, NOT price prediction, sentiment analysis, investment advice, or causal inference.

Your primary objective is to maximize the reliability of relevance labels and evidence. Do not maximize the number of articles classified as relevant.

A Federal Reserve source does not automatically make an article relevant to gold prices.

## 2. UNIT OF ANALYSIS
Analyze exactly one individual news release per evaluation.

The input may contain:

- Article title
- Article URL
- Source
- Publication date and time
- Article category
- Full article content
- Collection metadata
Evaluate the actual article content, using the title and metadata only as supporting context.

If the input contains an annual archive page listing multiple releases instead of one individual article, do not treat the entire archive as a single news release. Do not invent individual article content. The upstream data pipeline must first split the archive into individual releases.

If the content is missing, inaccessible, truncated, or insufficient to assess the article reliably, reflect that limitation in the classification and explanation. Never fabricate missing content.

## 3. DEFINITION OF GOLD-PRICE RELEVANCE
An article is relevant when its content provides a credible, explainable connection to gold prices through a direct relationship or a defensible economic transmission mechanism.

Evaluate the following mechanisms.

### A. Monetary policy and interest rates
- FOMC decisions and meeting statements
- Federal funds rate decisions
- Monetary policy guidance
- Rate-cut or rate-hike expectations
- Quantitative easing (QE) and quantitative tightening (QT)
- Federal Reserve balance-sheet policy
- Monetary policy projections, minutes, and speeches that contain substantive policy information

### B. Inflation and economic conditions
- Inflation-related information or expectations
- Employment and labor-market conditions, when linked to the Federal Reserve's policy outlook
- Economic outlooks that may affect expected monetary policy
- Evidence that changes expectations for future interest rates or real interest rates

### C. US dollar, Treasury yields, and financial conditions
- Changes or expectations concerning US Treasury yields or real yields
- US dollar-related developments, when supported by the article
- Liquidity conditions and changes in financial-market conditions
- Central-bank actions that may materially affect financial conditions

### D. Financial stability and risk sentiment
- Banking-sector stress
- Systemic financial risk
- Credit-market disruption
- Financial crises and emergency liquidity facilities
- Other material developments that could plausibly change demand for gold as a safe-haven asset

### E. Other defensible mechanisms
An article may be relevant through another mechanism only when the connection to gold prices is specific and economically defensible.

Do not force an article into a category simply because it comes from the Federal Reserve.

## 4. CLASSIFICATION LABELS
Assign exactly one label.

### relevant
Use when the article contains substantive information that has a clear, defensible connection to gold-price drivers.
Examples include an FOMC interest-rate decision, a substantive monetary-policy outlook, a significant change in Federal Reserve balance-sheet policy, or a material financial-stability announcement with a credible gold-market transmission mechanism. The article does not need to mention gold explicitly.

### uncertain
Use when a connection to gold prices is plausible but the available evidence is incomplete, ambiguous, indirect, or too weak to justify a confident relevant classification.
Examples include a broad economic discussion with unclear implications for monetary policy, a financial-sector development whose potential market implications are not established in the article, or content too incomplete to assess reliably. Do not use uncertain merely because the eventual direction of gold prices is unknown. A clear monetary-policy announcement can be relevant even when its price impact is uncertain.

### not_relevant
Use when the article has no sufficiently supported connection to gold-price drivers.
Examples include routine administrative notices, personnel announcements without substantive monetary-policy implications, procurement/operational/procedural notices, regulatory or legal announcements with no credible connection to gold prices, and general institutional announcements without meaningful economic or financial information. Do not classify an article as relevant solely because it mentions the Federal Reserve, the economy, banks, money, or financial regulation.

## 5. CONFIDENCE SCORING
Return an integer confidence score from 0 to 100. Confidence represents confidence in the assigned relevance label, NOT the probability gold prices rise or fall, the probability the article moves the market, future prediction accuracy, or statistical significance of a market reaction.

- 90–100: strongly supported by explicit article evidence and a clear relevance decision.
- 75–89: well supported, with minor ambiguity.
- 50–74: plausible but with meaningful ambiguity or limitations.
- 25–49: weakly supported because evidence is insufficient or unclear.
- 0–24: article cannot be assessed reliably from available content.

These are calibration guidelines, not mechanical rules. Choose the label first based on evidence, then assign confidence that reflects evidence quality and completeness. For incomplete or unavailable content, do not assign high confidence to a substantive relevance judgment without adequate support. A high-confidence not_relevant classification is appropriate when the full article has been examined and clearly lacks a defensible gold-price connection; it is not appropriate merely because the title does not mention gold.

## 6. REASON REQUIREMENTS
The reason must explain why the assigned label is appropriate. Write a concise explanation in Thai.
- relevant: identify actual disclosed information and explain its economic transmission mechanism to gold-price drivers; do not assert gold prices definitely rose or fell as a result.
- uncertain: explain the plausible connection and state what is ambiguous, indirect, or missing; do not invent a connection to avoid not_relevant.
- not_relevant: explain why available article content has no meaningful connection to gold-price drivers, not merely that it lacks the word gold.
Do not write generic reasons that could be copied to every article.

## 7. EVIDENCE REQUIREMENTS
The evidence field contains short, verbatim excerpts copied from supplied article content.
- Include only evidence present in input; preserve original language.
- Do not translate, paraphrase, reconstruct, or fabricate quoted evidence.
- Use up to three concise excerpts when useful, supporting the classification and mechanism without implying more than the source says.
- Do not cite external information as if it came from the article.
- If no usable supporting excerpt exists, return an empty array.
Evidence contains no explanations; explanations belong in reason.

## 8. TOPIC TAGGING
Use a normalized array of lowercase English snake_case topic tags, only for topics supported by actual article content. Preferred vocabulary:
monetary_policy, fomc, interest_rates, rate_expectations, inflation, employment, economic_outlook, real_yields, treasury_yields, usd, quantitative_easing, quantitative_tightening, fed_balance_sheet, liquidity, financial_stability, banking_stress, credit_conditions, safe_haven_demand, gold_price.
- Use zero or more tags; do not assign every possible tag to every relevant article.
- Do not use gold_price merely because the article is evaluated for gold relevance; use it only when the article explicitly discusses gold prices or their behavior.
- Use monetary_policy only when substantive monetary-policy content is present.
- Use an empty array when no meaningful topic is supported.

## 9. STRICT ANTI-HALLUCINATION RULES
Do not invent article facts, quotations, dates, or URLs; assume the Federal Reserve caused a gold-price change without evidence; call a decision hawkish/dovish unless supported; assume rate changes always move gold in one direction; assume financial instability always increases gold demand; infer market reactions from external historical patterns; use outside information to manufacture evidence; treat correlation, economic plausibility, and demonstrated causation as interchangeable; or confuse relevance with predictive accuracy. Qualify plausible but unestablished mechanisms in reason.

## 10. OUTPUT CONTRACT
Return exactly one valid JSON object and nothing else. No Markdown fences, comments, introductory text, or trailing explanations. Use exactly these five keys:
{"label":"relevant | uncertain | not_relevant","confidence":0,"reason":"คำอธิบายภาษาไทย","evidence":[],"topics":[]}
- label is exactly relevant, uncertain, or not_relevant.
- confidence is an integer 0–100.
- reason is a non-empty string in Thai.
- evidence is an array of verbatim excerpts from supplied article content, or an empty array.
- topics is an array of normalized topic tags, or an empty array.
- Use valid JSON with double-quoted keys/string values. Add no other fields. Return one object per article.

## 11. FINAL DECISION PRINCIPLE
Ask: Does this individual article provide sufficient evidence of a direct or defensible indirect connection to a factor that may influence gold prices? If yes, assign relevant. If a plausible connection exists but evidence or mechanism is insufficiently clear, assign uncertain. If no sufficiently supported connection exists, assign not_relevant.

When evidence is weak, do not force relevance. When evidence is strong, do not reject relevance merely because the article does not mention gold explicitly. Prioritize consistent, evidence-grounded classification that can be audited and used in a quantitative research pipeline.'''

ALLOWED_LABELS = {"relevant", "uncertain", "not_relevant"}
EXPECTED_CLASSIFICATION_KEYS = {"label", "confidence", "reason", "evidence", "topics"}
THAI_TEXT_RE = re.compile(r"[\u0e00-\u0e7f]")

print(f"System prompt loaded: {len(SYSTEM_PROMPT)} characters")

System prompt loaded: 10413 characters


In [8]:
def read_jsonl_strict(path):
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, raw_line in enumerate(handle, start=1):
            if not raw_line.strip():
                continue
            try:
                record = json.loads(raw_line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON at {path}:{line_number}: {exc}") from exc
            if not isinstance(record, dict):
                raise ValueError(f"Expected JSON object at {path}:{line_number}")
            records.append(record)
    return records


def stable_record_id(record):
    value = record.get("document_id")
    if value:
        return str(value)
    identity = "|".join(str(record.get(key) or "") for key in ("url", "title", "publication_date"))
    if not identity.strip("|"):
        raise ValueError("Record has no document_id or URL/title/publication_date fallback identity")
    return hashlib.sha256(identity.encode("utf-8")).hexdigest()


records = read_jsonl_strict(INPUT_PATH)
records_by_id = {}
for record in records:
    record_id = stable_record_id(record)
    if record_id in records_by_id:
        raise ValueError(f"Duplicate input record ID: {record_id}")
    records_by_id[record_id] = record
print(f"Loaded {len(records):,} unique source records")

Loaded 4,629 unique source records


In [9]:
def validate_classification(result, article_content):
    if not isinstance(result, dict) or set(result) != EXPECTED_CLASSIFICATION_KEYS:
        raise ValueError("Model output must contain exactly label, confidence, reason, evidence, topics")
    if result["label"] not in ALLOWED_LABELS:
        raise ValueError(f"Invalid classification label: {result['label']!r}")
    confidence = result["confidence"]
    if isinstance(confidence, bool) or not isinstance(confidence, int) or not 0 <= confidence <= 100:
        raise ValueError("confidence must be an integer from 0 to 100")
    reason = result["reason"]
    if not isinstance(reason, str) or not reason.strip() or not THAI_TEXT_RE.search(reason):
        raise ValueError("reason must be a non-empty Thai explanation")
    evidence = result["evidence"]
    if not isinstance(evidence, list) or len(evidence) > 3 or any(not isinstance(item, str) for item in evidence):
        raise ValueError("evidence must be an array of up to three strings")
    for quote in evidence:
        if not quote or quote not in article_content:
            raise ValueError("Evidence quote is empty or not a verbatim substring of article content")
    topics = result["topics"]
    if not isinstance(topics, list) or any(not isinstance(tag, str) or not re.fullmatch(r"[a-z][a-z0-9]*(?:_[a-z0-9]+)*", tag) for tag in topics):
        raise ValueError("topics must be an array of lowercase snake_case strings")
    return result


def parse_and_validate_response(raw_text, article_content):
    text = str(raw_text or "").strip()
    text = re.sub(r"^\s*```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```\s*$", "", text)
    text = re.sub(r"<think\b[^>]*>.*?</think\s*>", "", text, flags=re.IGNORECASE | re.DOTALL)
    decoder = json.JSONDecoder()
    validation_errors = []
    valid_results = []
    for match in re.finditer(r"\{", text):
        try:
            candidate, _ = decoder.raw_decode(text[match.start():])
        except json.JSONDecodeError:
            continue
        if not isinstance(candidate, dict):
            continue
        try:
            valid_results.append(validate_classification(candidate, article_content))
        except ValueError as exc:
            validation_errors.append(str(exc))
    if len(valid_results) == 1:
        return valid_results[0]
    if len(valid_results) > 1:
        raise ValueError("Model returned multiple schema-valid JSON objects")
    detail = validation_errors[-1] if validation_errors else "no JSON object found"
    raise ValueError(f"No schema-valid JSON object in model response: {detail}; preview={text[:240]!r}")


def article_prompt(record):
    return json.dumps({
        "title": record.get("title"),
        "url": record.get("url"),
        "source": record.get("source", "Federal Reserve Board"),
        "publication_date": record.get("publication_date"),
        "document_type": record.get("document_type"),
        "content": record.get("content") or "",
        "collection_metadata": {
            "document_id": record.get("document_id"),
            "publication_date_source": record.get("publication_date_source"),
            "source_archive_year": record.get("source_archive_year"),
        },
    }, ensure_ascii=False)


def append_jsonl(path, obj):
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(obj, ensure_ascii=False, allow_nan=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def load_completed_ids():
    if not OUTPUT_PATH.exists():
        return set()
    completed = set()
    for line_number, record in enumerate(read_jsonl_strict(OUTPUT_PATH), start=1):
        record_id = stable_record_id(record)
        if record_id not in records_by_id:
            raise ValueError(f"Existing output record is not in input: {record_id}")
        classification = record.get("classification")
        if not isinstance(classification, dict) or set(classification) != EXPECTED_CLASSIFICATION_KEYS:
            raise ValueError(f"Existing output has invalid classification for {record_id} (record {line_number})")
        if record_id in completed:
            raise ValueError(f"Duplicate completed record in output: {record_id}")
        completed.add(record_id)
    return completed


completed_ids = load_completed_ids()
pending_records = [(record_id, record) for record_id, record in records_by_id.items() if record_id not in completed_ids]
failures = {}
def write_progress_snapshot(status):
    with write_lock:
        snapshot = {
            "status": status,
            "source_records": len(records_by_id),
            "completed_records": len(completed_ids),
            "failed_this_run": len(failures),
            "pending_records": len(records_by_id) - len(completed_ids) - len(failures),
            "updated_at": datetime.now(timezone.utc).isoformat(),
            "output_jsonl": str(OUTPUT_PATH),
            "error_jsonl": str(ERROR_PATH),
        }
        temporary_path = PROGRESS_PATH.with_suffix(".tmp")
        with temporary_path.open("w", encoding="utf-8") as handle:
            json.dump(snapshot, handle, ensure_ascii=False, indent=2)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary_path, PROGRESS_PATH)


write_progress_snapshot("running")
print(f"Already completed: {len(completed_ids):,} | Pending: {len(pending_records):,} | Failed attempts will be recorded separately")

Already completed: 0 | Pending: 4,629 | Failed attempts will be recorded separately


In [ ]:
def classify_worker(client_index, assigned_records):
    client = clients[client_index]
    for record_id, record in assigned_records:
        if stop_event.is_set():
            break
        article_content = record.get("content") or ""
        last_error = None
        for attempt in range(1, MAX_RETRIES + 1):
            if stop_event.is_set():
                break
            try:
                if stop_event.wait(THAILLM_DELAY):
                    break
                response = client.chat.completions.create(
                    model=MODEL,
                    temperature=0,
                    max_tokens=1500,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": article_prompt(record)},
                    ],
                )
                raw_text = response.choices[0].message.content
                if not isinstance(raw_text, str) or not raw_text.strip():
                    raise ValueError(f"Model returned empty content (finish_reason={response.choices[0].finish_reason!r})")
                classification = parse_and_validate_response(raw_text, article_content)
                enriched = {
                    **record,
                    "classification": classification,
                    "classification_metadata": {
                        "provider": "ThaiLLM",
                        "model": MODEL,
                        "framework": "OpenAI Python SDK",
                        "classified_at": datetime.now(timezone.utc).isoformat(),
                        "prompt_version": PROMPT_VERSION,
                    },
                }
                with write_lock:
                    append_jsonl(OUTPUT_PATH, enriched)
                    completed_ids.add(record_id)
                    write_progress_snapshot_locked("running")
                last_error = None
                break
            except Exception as exc:
                last_error = f"{type(exc).__name__}: {exc}"
                if attempt < MAX_RETRIES and stop_event.wait(min(2 ** (attempt - 1), 8)):
                    break
        if stop_event.is_set() and record_id not in completed_ids:
            break
        if last_error is not None:
            with write_lock:
                append_jsonl(ERROR_PATH, {
                    "document_id": record_id,
                    "attempts": MAX_RETRIES,
                    "error": last_error,
                    "failed_at": datetime.now(timezone.utc).isoformat(),
                })
                failures[record_id] = last_error
                write_progress_snapshot_locked("running")
        completion_queue.put((record_id, last_error))


def write_progress_snapshot_locked(status):
    snapshot = {
        "status": status,
        "source_records": len(records_by_id),
        "completed_records": len(completed_ids),
        "failed_this_run": len(failures),
        "pending_records": len(records_by_id) - len(completed_ids) - len(failures),
        "updated_at": datetime.now(timezone.utc).isoformat(),
        "output_jsonl": str(OUTPUT_PATH),
        "error_jsonl": str(ERROR_PATH),
    }
    temporary_path = PROGRESS_PATH.with_suffix(".tmp")
    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(snapshot, handle, ensure_ascii=False, indent=2)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary_path, PROGRESS_PATH)


failures.clear()
stop_event.clear()
worker_batches = [[] for _ in API_KEYS]
for record_id, record in pending_records:
    key_index = int(hashlib.sha256(record_id.encode("utf-8")).hexdigest(), 16) % len(API_KEYS)
    worker_batches[key_index].append((record_id, record))
if len(pending_records) >= len(API_KEYS) and any(not batch for batch in worker_batches):
    raise RuntimeError("Expected pending records to use all seven API-key workers")
executor = ThreadPoolExecutor(max_workers=len(API_KEYS))
worker_futures = [
    executor.submit(classify_worker, index, batch)
    for index, batch in enumerate(worker_batches)
    if batch
]
interrupted = False
progress_bar = tqdm(total=len(pending_records), desc=f"Classifying with {len(worker_futures)} API workers")
try:
    while any(not future.done() for future in worker_futures):
        try:
            completion_queue.get(timeout=0.5)
            progress_bar.update(1)
        except Empty:
            pass
except KeyboardInterrupt:
    interrupted = True
    stop_event.set()
    print("Interrupt received. Finishing active requests and preserving completed JSONL records...")
finally:
    executor.shutdown(wait=True, cancel_futures=True)
    while not completion_queue.empty():
        completion_queue.get()
        progress_bar.update(1)
    progress_bar.close()

for future in worker_futures:
    future.result()
run_status = "interrupted" if interrupted else "complete"
write_progress_snapshot(run_status)
print(f"Run status: {run_status} | completed: {len(completed_ids):,}/{len(records_by_id):,} | failed this run: {len(failures):,}")

Classifying with 7 API workers:   0%|          | 0/4629 [00:00<?, ?it/s]

Run status: complete | completed: 4,553/4,629 | failed this run: 76


In [ ]:
final_completed_ids = load_completed_ids()
failed_ids = set(failures)
unaccounted_ids = set(records_by_id) - final_completed_ids - failed_ids
print(f"Source records: {len(records_by_id):,}")
print(f"Classified records: {len(final_completed_ids):,}")
print(f"Failed this run: {len(failed_ids):,}")
print(f"Pending after this run: {len(unaccounted_ids):,}")
print(f"Classified output: {OUTPUT_PATH}")
print(f"Error log: {ERROR_PATH}")
print(f"Progress checkpoint: {PROGRESS_PATH}")
if len(final_completed_ids) + len(failed_ids) + len(unaccounted_ids) != len(records_by_id):
    raise RuntimeError("Progress accounting mismatch")

Source records: 4,629
Classified records: 4,553
Failed this run: 76
Pending after this run: 0
Classified output: D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned\fed_press_releases_classified.jsonl
Error log: D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned\fed_press_releases_classification_errors.jsonl
Progress checkpoint: D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned\fed_press_releases_classification_progress.json
